In [ ]:
import os
from pathlib import Path

ROOT = Path("..").resolve()

for folder in [
    ROOT / "data" / "raw",
    ROOT / "data" / "processed",
    ROOT / "data" / "yolo"
]:
    print("\n", folder)

    for root, dirs, files in os.walk(folder):
        img_count = len([
            f for f in files
            if f.lower().endswith((".jpg",".jpeg",".png"))
        ])

        if img_count > 0:
            print(root, img_count)

경로 확인

In [ ]:
import os
from pathlib import Path

ROOT = Path("..").resolve()
NORMAL_DIR = ROOT / "data" / "processed" / "images" / "normal"
ABNORMAL_DIR = ROOT / "data" / "processed" / "images" / "abnormal"

normal_imgs = list(NORMAL_DIR.glob("*"))
abnormal_imgs = list(ABNORMAL_DIR.glob("*"))

normal_imgs = [p for p in normal_imgs if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
abnormal_imgs = [p for p in abnormal_imgs if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]

print("normal:", len(normal_imgs))
print("abnormal:", len(abnormal_imgs))

print("ROOT:", ROOT)
print("NORMAL_DIR:", NORMAL_DIR)
print("ABNORMAL_DIR:", ABNORMAL_DIR)
print("exists:", NORMAL_DIR.exists())

랜덤 후보 100장 추출

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

def show_images(paths, title="", cols=5):
    rows = (len(paths) + cols - 1) // cols
    plt.figure(figsize=(18, rows * 3))
    plt.suptitle(title, fontsize=18)

    for i, p in enumerate(paths):
        img = Image.open(p)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(p.name, fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

normal_sample = random.sample(normal_imgs, 50)
abnormal_sample = random.sample(abnormal_imgs, 80)

show_images(normal_sample, "Normal candidates")
show_images(abnormal_sample, "Abnormal candidates")

In [ ]:
import cv2
import pandas as pd

def analyze_images(paths, label):
    rows = []

    for path in paths:
        img = cv2.imread(str(path))
        if img is None:
            continue

        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        rows.append({
            "label": label,
            "file": path.name,
            "path": str(path),
            "brightness": gray.mean(),
            "edge_density": cv2.Canny(gray, 100, 200).mean(),
            "width": w,
            "height": h,
            "aspect_ratio": w / h,
        })

    return rows

df = pd.DataFrame(
    analyze_images(normal_imgs, "normal") +
    analyze_images(abnormal_imgs, "abnormal")
)

display(df.head())

1~3번 → normal 후보에서 고르기
4~6번, 10번 → abnormal 후보에서 고르기
7번 → night 후보에서 고르기
8번 → complex background 후보에서 고르기
9번 → abnormal/normal 후보 중 킥보드 여러 대 보이는 것 수동 선택

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

def show_images(paths, title="", cols=5):
    paths = list(paths)

    rows = (len(paths) + cols - 1) // cols

    plt.figure(figsize=(18, rows * 3))
    plt.suptitle(title, fontsize=18)

    for i, p in enumerate(paths):

        p = Path(p)   # 추가

        img = Image.open(p)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(p.name, fontsize=8)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
print("야간 후보")
night = df.sort_values("brightness").head(30)
display(night[["label", "file", "brightness"]])
show_images(night["path"], "Night candidates")

print("복잡한 배경 후보")
complex_bg = df.sort_values("edge_density", ascending=False).head(30)
display(complex_bg[["label", "file", "edge_density"]])
show_images(complex_bg["path"], "Complex background candidates")

print("정상 후보")
normal_candidates = df[df["label"] == "normal"].sample(50)
show_images(normal_candidates["path"], "Normal candidates")

print("abnormal 후보")
abnormal_candidates = df[df["label"] == "abnormal"].sample(80)
show_images(abnormal_candidates["path"], "Abnormal candidates")